# CH4 marginality: exact-year UNL budgets

Nebraska MLRA 106 dryland corn; odd years 2001–2021. Original yields are read-only. **2021 is experimental.** See METHODS.md and the eligibility register. Economics is blocked for every year without a matching original UNL budget. Install requirements.txt if your runtime lacks the recorded dependencies.

In [ ]:
from pathlib import Path
import sys, importlib.metadata, json
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
from google.colab import drive, auth
if not ROOT.exists():
 try:
  drive.mount('/content/drive',force_remount=False)
 except Exception:
  auth.authenticate_user()
  drive.mount('/content/drive',force_remount=True)
PACKAGE=ROOT/'CH4_marginality/20260928_corefilter_exact_year_v2/code'
sys.path.insert(0,str(PACKAGE))
import importlib.util
print('Google Colab runtime available:', importlib.util.find_spec('google.colab') is not None)
print({p:importlib.metadata.version(p) for p in ['numpy','pandas','rasterio','geopandas','scipy','nbclient']})

In [ ]:
import unittest
suite=unittest.defaultTestLoader.discover(str(PACKAGE/'ch4_marginality/tests'))
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

In [ ]:
import ee
try:
    ee.Initialize(project='ee-njberkowitz95')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-njberkowitz95')
print('Earth Engine authenticated; prior CH4 asset bands:', ee.Image('projects/ee-njberkowitz95/assets/ch4_verified_yields_20260928').bandNames().size().getInfo())

In [ ]:
from ch4_marginality.verify_ee_assets import run as verify_earth_engine
from ch4_marginality.verify_app_classes import run as verify_earth_engine_classes
OUTPUT=ROOT/'CH4_marginality/20260928_corefilter_exact_year_v2'
verify_earth_engine(OUTPUT)
verify_earth_engine_classes(OUTPUT)

In [ ]:
from ch4_marginality.pipeline import run
OUTPUT=run(ROOT)
print('Completed output:',OUTPUT)

In [ ]:
%pip -q install reportlab markdown beautifulsoup4
from ch4_marginality.final_checks import verify, pdf_report
validation=verify(OUTPUT)
pdf_report(OUTPUT)
print(json.dumps(validation,indent=2))

In [ ]:
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(OUTPUT/'tables/year_eligibility.csv'))
display(pd.read_csv(OUTPUT/'tables/annual_yield.csv'))
s=pd.read_csv(OUTPUT/'tables/sensitivity.csv')
display(s[(s.price_factor==1)&(s.cost_factor==1)])

In [ ]:
display(Image(filename=str(OUTPUT/'figures/overlap_maps_2019.png')))
display(Image(filename=str(OUTPUT/'figures/overlap_maps_2021.png')))
print('Experimental 2021 excluded from primary temporal summaries.')

Outputs include the workbook, source ledger, CSV tables, GeoPackage, COGs, publication figures and methods/results report. FINBIN is an operator-account proxy and does not establish complete economic profitability. Missing data never become zero.